# 04 — Scalable Oversight

Scalable oversight addresses how humans can verify AI behaviour when the AI is more capable than the human at the task.

## Debate, Amplification, and Recursive Reward Modelling

**The oversight problem**: as AI systems become more capable, human evaluators may not be able to verify correctness. A human cannot evaluate whether a proof of a hard theorem is correct, whether a long piece of code is secure, or whether a complex economic policy is sound.

**Debate** (Irving et al., 2018):
- Two AI agents argue for opposite answers
- A human judge evaluates the debate, not the underlying task
- Key claim: honest agents can always defeat dishonest agents in debate because a lie can be refuted given enough rounds
- Problem: requires the human to identify the strongest counter-argument, not just evaluate answers

**Iterated Amplification** (Christiano et al., 2018):
- Start with a weak, human-interpretable oracle H
- Build a stronger oracle by having H decompose the task into sub-tasks it can solve
- Iteratively amplify: use the stronger oracle to train a new model, then amplify again
- Each level produces a supervisor that is stronger than the previous but remains aligned

**Recursive Reward Modelling** (Leike et al., 2018):
- Train a reward model R1 from human preferences
- Use R1 to help humans evaluate harder tasks → R2
- Iterate until reward models can evaluate tasks humans couldn't directly

In [ ]:
# Debate protocol implementation
import numpy as np

class DebateProtocol:
    """
    Simulated debate between honest and dishonest agents.
    Each agent makes claims; a judge evaluates.
    """

    def __init__(self, n_rounds=3, judge_accuracy=0.7):
        self.n_rounds = n_rounds
        self.judge_accuracy = judge_accuracy

    def honest_agent(self, question, round_num):
        """Honest: makes correct claims, can refute dishonest claims."""
        if round_num == 0:
            return f'The correct answer is A because fact_1 and fact_2 support it.'
        else:
            return f'Round {round_num}: The opponent\'s claim is false. Here is the refutation: counter_fact_{round_num}'

    def dishonest_agent(self, question, round_num):
        """Dishonest: makes plausible but false claims."""
        if round_num == 0:
            return f'The answer is actually B. Consider misleading_fact_{round_num}.'
        else:
            return f'Round {round_num}: My reasoning is compelling because spurious_claim_{round_num}'

    def judge(self, honest_claim, dishonest_claim, correct_answer_is_honest=True):
        """Judge evaluates debate with some accuracy."""
        if correct_answer_is_honest:
            return np.random.random() < self.judge_accuracy
        else:
            return np.random.random() > self.judge_accuracy

    def run_debate(self, question, correct_answer='A'):
        """Run a full debate and return winner."""
        honest_points = 0
        dishonest_points = 0

        for round_num in range(self.n_rounds):
            h_claim = self.honest_agent(question, round_num)
            d_claim = self.dishonest_agent(question, round_num)

            # Judge evaluates this round
            honest_wins_round = self.judge(h_claim, d_claim)
            if honest_wins_round:
                honest_points += 1
            else:
                dishonest_points += 1

        winner = 'honest' if honest_points > dishonest_points else 'dishonest'
        return winner, honest_points, dishonest_points

# Simulate many debates
protocol = DebateProtocol(n_rounds=5, judge_accuracy=0.7)
n_debates = 1000
honest_wins = 0

np.random.seed(42)
for _ in range(n_debates):
    winner, h_pts, d_pts = protocol.run_debate('complex question')
    if winner == 'honest':
        honest_wins += 1

print(f'Debate results over {n_debates} rounds:')
print(f'  Honest agent wins: {honest_wins} ({honest_wins/n_debates:.1%})')
print(f'  Dishonest agent wins: {n_debates - honest_wins} ({(n_debates-honest_wins)/n_debates:.1%})')
print(f'  Judge accuracy: {protocol.judge_accuracy:.0%}')

# Effect of number of rounds
round_counts = range(1, 11)
win_rates = []
for n_r in round_counts:
    p = DebateProtocol(n_rounds=n_r, judge_accuracy=0.65)
    wins = sum(1 for _ in range(500) if p.run_debate('q')[0] == 'honest')
    win_rates.append(wins / 500)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(round_counts, win_rates, 'o-', color='steelblue')
ax.axhline(0.5, color='gray', linestyle='--', label='Random')
ax.set_xlabel('Number of debate rounds')
ax.set_ylabel('Honest agent win rate')
ax.set_title('Debate Protocol: More Rounds Help Honest Agents')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/debate.png', dpi=100, bbox_inches='tight')
plt.show()
print('Debate protocol demo complete')